# Limpieza de los datos
En este notebook se realizara la limpieza siguiendo el análisis obtenido en el archivo eda_data.ipynb. Durante el análisis se ideintificaron las siguiente anomalías:
- Registros duplicados
- Transacciones sin identificación
- Cantidades y precios unitarios negativos
- Outliers en cantidades y precios unitarios

## 0. Imports y Carga del Dataset

Preparamos el entorno de trabajo: importamos librerías, configuramos la estética visual, definimos las rutas del proyecto y cargamos el dataset original. A continuación creamos las columnas auxiliares necesarias para los pasos de limpieza (`Fecha`, `Mes`, `DiaSemana`, `EsCancelacion`, `TotalPrice`) e inicializamos el diccionario de auditoría `stats_cleaning` que registrará paso a paso cuántos registros eliminamos y por qué.


In [3]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# Configuración visual global
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)
%matplotlib inline

# Rutas del proyecto
RUTA_CSV      = '../../../data/raw/data.csv'
RUTA_GRAFICOS = '../../../graphics/'
RUTA_INTERIM  = '../../../data/interim/'
os.makedirs(RUTA_GRAFICOS, exist_ok=True)
os.makedirs(RUTA_INTERIM,  exist_ok=True)

print('Librerías cargadas correctamente.')


Librerías cargadas correctamente.


In [4]:
# Carga del dataset original
df_raw     = pd.read_csv(RUTA_CSV, encoding='latin-1')
df_working = df_raw.copy()

# ── Columnas auxiliares ────────────────────────────────────────────────────────
df_working['InvoiceDate']   = pd.to_datetime(df_working['InvoiceDate'], format='mixed')
df_working['Fecha']         = df_working['InvoiceDate'].dt.normalize()
df_working['Mes']           = df_working['InvoiceDate'].dt.to_period('M')
df_working['DiaSemana']     = df_working['InvoiceDate'].dt.day_name()
df_working['EsCancelacion'] = df_working['InvoiceNo'].str.startswith('C')
df_working['TotalPrice']    = df_working['Quantity'] * df_working['UnitPrice']

# ── Diccionario de auditoría (se actualiza en cada paso) ──────────────────────
stats_cleaning = {'Registros Iniciales': len(df_raw)}

# ── Resumen de carga ──────────────────────────────────────────────────────────
print('=' * 55)
print(f'  DATASET CARGADO')
print('=' * 55)
print(f'  Filas    : {df_raw.shape[0]:>10,}')
print(f'  Columnas : {df_raw.shape[1]:>10}')
print('=' * 55)
print(f'\n  df_working activo : {len(df_working):,} filas')
print(f'  Columnas auxiliares añadidas:')
print(f'    - Fecha, Mes, DiaSemana, EsCancelacion, TotalPrice')


  DATASET CARGADO
  Filas    :    541,909
  Columnas :          8

  df_working activo : 541,909 filas
  Columnas auxiliares añadidas:
    - Fecha, Mes, DiaSemana, EsCancelacion, TotalPrice


# 3. LIMPIEZA DE DATOS

### 3.1 ELIMINAR FILAS CON Description NULA

Motivo: el 100% de estas filas cumplen simultáneamente:
 - Description = NaN  → no sabemos qué producto es
 - UnitPrice = 0      → no generan ningún ingreso (TotalPrice = 0)
 - CustomerID = NaN   → no tienen cliente asociado
No son recuperables y solo añadirían ruido al modelo.

In [5]:
# 3.1 — Eliminar filas con Description nula
antes = len(df_working)
df_working = df_working.dropna(subset=['Description']).reset_index(drop=True)
eliminadas_desc = antes - len(df_working)
stats_cleaning['Description nulas eliminadas'] = eliminadas_desc

print(f"── 3.1 Eliminar filas con Description nula ──")
print(f"  Justificación: el 100% tienen UnitPrice=0 y CustomerID=NaN simultáneamente")
print(f"  → no generan ingreso ni tienen cliente → ruido puro para el modelo")
print()
print(f"  Filas antes     : {antes:,}")
print(f"  Filas eliminadas: {eliminadas_desc:,}")
print(f"  Filas después   : {len(df_working):,}")
print(f"  Verificación    — Description nulos restantes: {df_working['Description'].isnull().sum()}")


── 3.1 Eliminar filas con Description nula ──
  Justificación: el 100% tienen UnitPrice=0 y CustomerID=NaN simultáneamente
  → no generan ingreso ni tienen cliente → ruido puro para el modelo

  Filas antes     : 541,909
  Filas eliminadas: 1,454
  Filas después   : 540,455
  Verificación    — Description nulos restantes: 0


### 3.2 Eliminar duplicados exactos

Una fila duplicada exacta tiene **todos** sus campos idénticos: mismo `InvoiceNo`, `StockCode`, `Description`, `Quantity`, `UnitPrice`, `InvoiceDate`, `CustomerID` y `Country`. Esto es físicamente imposible en un sistema transaccional real — su presencia indica errores de doble inserción en la base de datos, exports corruptos o fallos de ETL. Se conserva la primera ocurrencia (`keep='first'`) al ser todas idénticas.


In [ ]:
antes = len(df_working)
df_working = df_working.drop_duplicates(keep='first', ignore_index=True)
eliminadas_dup = antes - len(df_working)
stats_cleaning['Duplicados eliminados'] = eliminadas_dup

print(f"── 3.2 Eliminar duplicados exactos ──")
print(f"  Filas antes     : {antes:,}")
print(f"  Filas eliminadas: {eliminadas_dup:,}")
print(f"  Filas después   : {len(df_working):,}")
print(f"  Verificación    — duplicados restantes: {df_working.duplicated().sum()}")


In [ ]:
# [DOC] Habiendo eliminado los duplicados y nulos, filtramos los datos para mantener solo cantidades y precios mayores a cero
df_working = df_working[(df_working['Quantity'] > 0) & (df_working['UnitPrice'] > 0)]
print(f"[INFO] Registros tras eliminar transacciones no comerciales: {df_working.shape[0]}")

In [ ]:
# [DOC] Realicamos el cálculo de límites basados en el percentil 99 y aplicamos el filtro de outliers

q_limit = df_working['Quantity'].quantile(0.99)
p_limit = df_working['UnitPrice'].quantile(0.99)
df_working = df_working[(df_working['Quantity'] <= q_limit) & (df_working['UnitPrice'] <= p_limit)]

stats_cleaning["Registros Finales"] = len(df_working)

print(f"[INFO] Límite de Quantity aplicado: {q_limit}")
print(f"[INFO] Límite de UnitPrice aplicado: {p_limit}")
print(f"[INFO] Registros tras limpieza de outliers: {df_working.shape[0]}")

In [ ]:
# [DOC] Comenzamos a verificar la integridad de los datos una vez realizada la limpieza, primero generamos un heatmap

plt.figure(figsize=(10, 3))
sns.heatmap(df_working.isnull(), yticklabels=False, cbar=False, cmap='viridis')
plt.title('Verificación Final: Mapa de Presencia de Datos')
plt.show()

In [ ]:
# [DOC] Ahora generamos comparativas entre los datos brutos y los datos limpios de los campos de cantidad y precio unitario

fig, ax = plt.subplots(2, 2, figsize=(16, 10))

# [DOC] Histograma para Quantity
sns.histplot(df_raw['Quantity'], bins=50, ax=ax[0,0], color='lightgrey', kde=False)
ax[0,0].set_title('Quantity: Distribución Original')
ax[0,0].set_yscale('log')

sns.histplot(df_working['Quantity'], bins=50, ax=ax[0,1], color='skyblue', kde=True)
ax[0,1].set_title('Quantity: Distribución Saneada')

# [DOC] Histograma para UnitPrice
sns.histplot(df_raw['UnitPrice'], bins=50, ax=ax[1,0], color='lightgrey', kde=False)
ax[1,0].set_title('UnitPrice: Distribución Original')
ax[1,0].set_yscale('log')

sns.histplot(df_working['UnitPrice'], bins=50, ax=ax[1,1], color='salmon', kde=True)
ax[1,1].set_title('UnitPrice: Distribución Saneada')

plt.tight_layout()
plt.show()

In [ ]:
# [DOC] Esta última visualización es un balance de datos entre el dataset original y el dataset saneado

labels = ['Datos Conservados', 'Datos Eliminados (Ruido)']
sizes = [len(df_working), len(df_raw) - len(df_working)]

plt.figure(figsize=(7, 7))
plt.pie(sizes, labels=labels, autopct='%1.1f%%', startangle=140, colors=['#66b3ff','#ff9999'], explode=(0.1, 0))
plt.title('Balance Final del Proceso de Saneamiento')
plt.show()

In [ ]:
# [DOC] Por último, exportamos los datos saneados y limpios

output_dir = '../../data/interim'
output_path = os.path.join(output_dir, 'data_sanitized.csv')

if not os.path.exists(output_dir):
    os.makedirs(output_dir)
    print(f"[INFO] Carpeta creada: {output_dir}")

df_working.to_csv(output_path, index=False)

print(f"[SUCCESS] Proceso finalizado. Dataset guardado en: {output_path}")
print(f"[INFO] Resumen técnico: {stats_cleaning}")